# KITTI queue runtime (A or B)

Run top to bottom on an **L4** runtime. Keep this tab open overnight: Colab Pro can drop an idle session.

- **Runtime A:** every cell. The smoke test and the epoch decision happen here, once.
- **Runtime B (two-runtime mode only):** cells 1–5, then cell 8 after `EPOCH_DECISION.json` exists.
- **After a crash:** rerun cells 1–5, then cell 8. Finished runs are skipped and an interrupted run resumes from `last.pt`.

In [ ]:
# 1. Settings
RUNTIME = "A"          # "A" on the first runtime, "B" on the second
TAG = "prereg-v1"      # the emailed pre-registration tag
DRIVE = "/content/drive/MyDrive/msc"
DATA, RAW = "/content/kitti", "/content/kitti_raw"
LABEL_ZIP_SHA = "4efc76220d867e1c31bb980bbf8cbc02599f02a9cb4350effa98dbb04aaed880"  # zip the split was built from
DEVKIT_ZIP_SHA = "ce0b76b69c0c5f89690a0d65b7302bbbdb962a0c7e8aba6efc7050d1b04b4cf1"

In [ ]:
# 2. Drive and GPU
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p {DRIVE}/day1 {DRIVE}/runs
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv | tee {DRIVE}/day1/gpu_{RUNTIME}.txt

In [ ]:
# 3. Code at the tag, pinned Ultralytics
!rm -rf /content/code && git clone -q {DRIVE}/code.bundle /content/code && git -C /content/code checkout -q {TAG} && git -C /content/code log -1 --format='%H %d'
%cd /content/code
!pip install -q ultralytics==8.4.171
import ultralytics, torch
print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| CUDA", torch.version.cuda, "|", torch.cuda.get_device_name(0))
assert ultralytics.__version__ == "8.4.171"

In [ ]:
# 4. Official KITTI files (about 5 min); zip hashes logged and checked against the split's labels
!mkdir -p {RAW}
%cd {RAW}
!for f in data_object_image_2.zip data_object_label_2.zip devkit_object.zip; do [ -f $$f ] || wget -q https://s3.eu-central-1.amazonaws.com/avg-kitti/$$f; done
!sha256sum *.zip | tee {DRIVE}/day1/zips_sha256_{RUNTIME}.txt
sums = dict(l.split()[::-1] for l in open(f"{DRIVE}/day1/zips_sha256_{RUNTIME}.txt"))
assert sums["data_object_label_2.zip"] == LABEL_ZIP_SHA and sums["devkit_object.zip"] == DEVKIT_ZIP_SHA, sums
!unzip -qo data_object_image_2.zip 'training/*' && unzip -qo data_object_label_2.zip && ls training && ls training/image_2 | wc -l
%cd /content/code

In [ ]:
# 5. Day-1 gate: KITTI -> YOLO labels, 1 px round trip on all 7,481 files, 004696, overlay sheets
!python scripts/convert_labels.py {RAW}/training/label_2 {RAW}/training/image_2 {DATA} {DRIVE}/day1/roundtrip_{RUNTIME}.json --overlay splits {DRIVE}/day1/overlay_{RUNTIME} | tail -25

In [ ]:
# 6. Runtime A only: smoke test (s 3 epochs; n, m, s@960 1 epoch each) and queue projections
assert RUNTIME == "A"
!python scripts/smoke_test.py --data-root {DATA} --runs-root {DRIVE}/runs 2>&1 | tee {DRIVE}/day1/smoke_console.log | grep -E "s/epoch|queue_hours|sec_per_epoch|Error|error" ; cat {DRIVE}/runs/smoke/smoke_projection.json

In [ ]:
# 7. Runtime A only, ONCE: record the epoch decision (PREREG 3.3). Edit the three values first.
#    two-runtime: runtime B is open, both are L4, compute-unit balance >= ~30  -> 100 epochs, queues A + B
#    single:      otherwise -> 50 epochs, one queue
MODE, GPU_B, CU_BALANCE = "two-runtime", "NVIDIA L4", 0
gpu_a = torch.cuda.get_device_name(0)
!python scripts/decide_epochs.py --runs-root {DRIVE}/runs --mode {MODE} --gpu-a "{gpu_a}" --gpu-b "{GPU_B}" --cu-balance {CU_BALANCE}

In [ ]:
# 8. Launch this runtime's queue (runs for hours; keep the tab open)
import json
decision = json.load(open(f"{DRIVE}/runs/EPOCH_DECISION.json"))
QUEUE = RUNTIME if decision["mode"] == "two-runtime" else "single"
print("epochs", decision["epochs"], "| queue", QUEUE, decision["queues"][QUEUE])
!python scripts/train_queue.py --queue {QUEUE} --data-root {DATA} --runs-root {DRIVE}/runs 2>&1 | tee -a {DRIVE}/runs/console_{QUEUE}.log